In [68]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd


In [69]:
def create_webdriver():
    opts = Options()
    # opts.add_argument("--headless=new")  # enable for no-GUI scraping
    # opts.add_experimental_option("detach", True)  # keep window open after script ends
    return webdriver.Chrome(options=opts)


WEBSITE = "https://old.reddit.com/r/wallstreetbets/"
# https://old.reddit.com/r/wallstreetbets/
# https://books.toscrape.com/
driver = create_webdriver()
driver.get(WEBSITE)


In [70]:
# Wait until the project links are present
titles = []
post_idList = []
created_utcList = []
flairsList = []
# upvotesList = []
num_commentsList = []
permalinkList = []

i = 0

while i < 10:
    wait = WebDriverWait(driver, 10)
    things = wait.until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, "div.thing"))
    )




    for thing in things:
        if "stickied" in thing.get_attribute("class"):
            continue
        if thing.get_attribute("data-promoted") == "true":
            continue
        title = thing.find_element(By.CSS_SELECTOR, 'p.title > a')
        titles.append(title.text)

        post_id = thing.get_attribute("data-fullname")
        post_idList.append(post_id)

        time_elements = thing.find_elements(By.TAG_NAME, "time")
        if time_elements:
            created_utc = time_elements[0].get_attribute("datetime")
        else:
            created_utc = None
        created_utcList.append(created_utc)

        flairs = thing.find_element(By.CSS_SELECTOR, 'span.linkflairlabel')
        flairsList.append(flairs.text)

        # upvotes = thing.find_element(By.CSS_SELECTOR, 'div.score')
        # upvotesList.append(upvotes.text)

        comments = thing.find_element(By.CSS_SELECTOR, 'a.comments')
        num_commentsList.append(comments.text)

        permalink = comments.get_attribute("href")
        permalinkList.append(permalink)

        


    
    next_button = WebDriverWait(driver, 10).until(EC.presence_of_element_located(
        (By.CSS_SELECTOR, 'span.next-button')))
    if next_button.get_attribute('Disabled'):
        break  # Exit the loop if the next button is disabled
    else:
        # Click the next button to navigate to the next page
        next_button.click()
    i += 1


driver.quit()


In [71]:
df = pd.DataFrame({'titles': titles, 'post_idList': post_idList, 'created_utc': created_utcList, 'flairs': flairsList, 'num_comments': num_commentsList, 'permalink': permalinkList})
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 500)  # adjust for your screen
pd.set_option('display.max_columns', None)
print(df)

                                                                                                                                    titles post_idList                created_utc              flairs    num_comments                                                                                                     permalink
0                                                                                     The real price is the friends we made along the way.  t3_1n0nvtw  2025-08-26T14:48:44+00:00                Meme     45 comments     https://old.reddit.com/r/wallstreetbets/comments/1n0nvtw/the_real_price_is_the_friends_we_made_along_the/
1                                                                                                            Trump fired Fed Gov Lisa Cook  t3_1n07etr  2025-08-26T00:23:11+00:00                News   1164 comments                       https://old.reddit.com/r/wallstreetbets/comments/1n07etr/trump_fired_fed_gov_lisa_cook/
2                           

# Next Steps:
- data cleaning
- perform analytics
- visualizations

In [72]:
df['titles'] = df['titles'].str.replace('\n', ' ', regex=False).str.strip()
df['flairs'] = df['flairs'].fillna('').str.strip()
df = df[df['flairs'] != 'Daily Discussion']

In [73]:
df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))
df['num_comments']

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
/var/folders/49/dj3znqqj2zx9gx86y4p9jzgr0000gq/T/ipykernel_24056/1112965993.py:1: SyntaxWarning: invalid escape sequence '\d'
  df['num_comments'] = (df['num_comments'].str.extract('(\d+)').fillna(0).astype(int))


0        45
1      1164
2        67
3        37
4        60
5        46
6       102
7        11
8       172
9       221
10      126
11       26
12       16
13      403
14       74
15      153
16       10
17       13
18        8
19      197
21       85
22       35
23       69
24       14
25      312
26       82
27       34
28       32
29       30
30       83
31       85
32       26
33       48
35       26
36       78
37      171
38       29
39       89
40      268
41       11
42       37
43       79
44        9
45       20
46        9
47       40
48       98
49      958
51      197
52       19
53        8
54       61
55       19
56       43
57       24
58      486
59      147
60      329
61       57
62      379
63      113
64      342
65       68
66      718
67      214
68      151
69       47
70      110
71      761
72      529
73      107
74       42
75      185
76      140
77      261
78       47
79       82
80      256
81      448
82      168
83       94
84      149
85       37
86  

In [29]:
df['created_utc'] = pd.to_datetime(df['created_utc'], utc=True)
df['date'] = df['created_utc'].dt.date

In [74]:
pattern = r'\b[A-Z]{1,5}\b'
df['tickers'] = df['titles'].str.findall(pattern)
df['tickers']


0                                  []
1                                  []
2                             [AT, T]
3                              [OPEN]
4                                  []
5                                 [I]
6                         [NVDA, FED]
7                                  []
8                                  []
9                              [OPEN]
10                             [YOLO]
11                                 []
12                                 []
13                                 []
14                             [NVDA]
15                       [SNAP, YOLO]
16                             [DOMO]
17                             [RKLB]
18                              [CCL]
19                          [I, YOLO]
21                             [YOLO]
22                             [ULTY]
23                                [I]
24                             [RKLB]
25                             [OPEN]
26                             [OPEN]
27          

## need to grab nasdaq list and nyse list to compare ticker symbols with

In [66]:
df = pd.read_csv("nasdaqlist.txt", sep="|")
df2 = pd.read_csv("nasdaqlistpt2.txt", sep="|")

# print(df.columns)
# print(df['NASDAQ Symbol'])
# print(df2['Symbol'])
tickers = set(df['NASDAQ Symbol'])
tickers2 = set(df2['Symbol'])
tickers_all = tickers.union(tickers2)
# tickers_all

In [ ]:
for arr in df['tickers']:
	